# cAPTure: Stage-2 training preflight v2

This notebook freezes and audits bounded chronological validation blocks before any model training. For each development scenario, it selects the latest admissible 20% block, falling back to 25% only when necessary. It refuses to split attack-step iterations, excludes the later suffix during epoch selection, verifies graph/packet alignment, and runs inference-only synthetic model-interface checks.

It does not optimize model parameters, select score thresholds, or access Test1, Test2, `train_pub_exf`, or `train_user_prop`.

## 1. Mount Drive and prepare the environment

Push the current branch before opening Colab. Use a fresh runtime so imported modules cannot come from an older commit.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys

REPOSITORY_URL = "https://github.com/tatipar/temporalgnn-nids.git"
REPOSITORY_BRANCH = "feat/capture-feasibility"
PROJECT_ROOT = Path("/content/temporalgnn-nids")
DRIVE_ROOT = Path("/content/drive/MyDrive/capture_gate0")
LOCAL_GRAPH_PARENT = Path("/content/capture_graph_stage2_inputs_v2")
LOCAL_PREFLIGHT_WORK_ROOT = Path("/content/capture_graph_preflight_work")

repository_already_present = PROJECT_ROOT.exists()
if not repository_already_present:
    subprocess.run(
        ["git", "clone", "--branch", REPOSITORY_BRANCH, "--single-branch",
         REPOSITORY_URL, str(PROJECT_ROOT)],
        check=True,
    )
if repository_already_present:
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", REPOSITORY_BRANCH],
        cwd=PROJECT_ROOT,
        check=True,
    )
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=PROJECT_ROOT, text=True
).strip()
if branch != REPOSITORY_BRANCH:
    raise RuntimeError(f"Expected branch {REPOSITORY_BRANCH}, found {branch}.")

required_files = [
    "code/python/requirements-capture-mlp.txt",
    "code/python/utils/capture_graph_dataset.py",
    "code/python/utils/capture_graph_training_preflight.py",
    "code/python/utils/capture_graph_training_preflight_v2.py",
    "configs/capture_graph_stage2_input_v2.yaml",
    "configs/capture_graph_training_preflight_v2.yaml",
]
missing = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing:
    raise FileNotFoundError(f"Update the Colab repository copy first: {missing}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r",
     str(PROJECT_ROOT / "code/python/requirements-capture-mlp.txt")],
    check=True,
)
sys.path.insert(0, str(PROJECT_ROOT / "code/python"))

import pandas as pd
from IPython.display import display
from utils.capture_graph_dataset import stage_capture_graph_materialization
from utils.capture_graph_training_preflight_v2 import (
    load_completed_training_preflight_v2,
    run_capture_graph_training_preflight_v2,
    save_training_preflight_v2_decision,
)

print("Repository commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
).strip())
print("Training-preflight environment is ready.")

## 2. Bind the reviewed immutable inputs

The preflight contract names the exact prepared, graph-materialization, and graph-input-audit runs. A different run ID or checksum is rejected.

In [ ]:
PREPARED_RUN_ID = "20260917T235058_827743Z_prepare_full_dev"
GRAPH_MATERIALIZATION_RUN_ID = (
    "20260928T175821_521784Z_graph_materialization_full_dev"
)
GRAPH_INPUT_AUDIT_RUN_ID = "20260928T193308_017682Z_graph_input_full"

PREPARED_RUN_DIR = DRIVE_ROOT / "prepared_runs" / PREPARED_RUN_ID
GRAPH_MATERIALIZATION_DIR = (
    DRIVE_ROOT / "graph_materialization_runs" / GRAPH_MATERIALIZATION_RUN_ID
)
GRAPH_INPUT_AUDIT_DIR = (
    DRIVE_ROOT / "graph_input_audit_runs" / GRAPH_INPUT_AUDIT_RUN_ID
)
GRAPH_INPUT_CONTRACT_PATH = (
    PROJECT_ROOT / "configs/capture_graph_stage2_input_v2.yaml"
)
PREFLIGHT_CONFIG_PATH = (
    PROJECT_ROOT / "configs/capture_graph_training_preflight_v2.yaml"
)
for path in (
    PREPARED_RUN_DIR, GRAPH_MATERIALIZATION_DIR, GRAPH_INPUT_AUDIT_DIR,
    GRAPH_INPUT_CONTRACT_PATH, PREFLIGHT_CONFIG_PATH,
):
    if not path.exists():
        raise FileNotFoundError(f"Required preflight input is missing: {path}")

print("Prepared packets:", PREPARED_RUN_DIR)
print("Graph materialization:", GRAPH_MATERIALIZATION_DIR)
print("Graph-input audit:", GRAPH_INPUT_AUDIT_DIR)
print("Preflight contract:", PREFLIGHT_CONFIG_PATH)

## 3. Stage and verify the graph collection

The existing FULL_DEV materialization is copied to local Colab storage and every artifact checksum is verified. No graph is rebuilt.

In [ ]:
collection = stage_capture_graph_materialization(
    source_root=GRAPH_MATERIALIZATION_DIR,
    local_parent=LOCAL_GRAPH_PARENT,
    contract_path=GRAPH_INPUT_CONTRACT_PATH,
    expected_run_id=GRAPH_MATERIALIZATION_RUN_ID,
)
print("Verified local graph root:", collection.root)
print("All graph checksums verified:",
      collection.artifact_checksums_verified)

## 4. Run or resume the training preflight

This scans only prepared development metadata, finds the latest admissible bounded validation block for each scenario, verifies graph positions for the training, validation, and excluded suffix partitions, and performs inference-only synthetic forward/reset checks. It may take several minutes because the prepared artifacts are checksum-verified and staged one scenario at a time.

In [ ]:
PREFLIGHT_RUN_ID = None  # Paste an existing completed v2 ID to reuse it.
if PREFLIGHT_RUN_ID is None:
    PREFLIGHT_RUN_ID = (
        datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
        + "_graph_training_preflight_v2"
    )
PREFLIGHT_OUTPUT_DIR = (
    DRIVE_ROOT / "graph_training_preflight_v2_runs" / PREFLIGHT_RUN_ID
)
print("Training-preflight run ID:", PREFLIGHT_RUN_ID)
print("Training-preflight output:", PREFLIGHT_OUTPUT_DIR)

if PREFLIGHT_OUTPUT_DIR.exists():
    preflight_report = load_completed_training_preflight_v2(
        PREFLIGHT_OUTPUT_DIR,
        collection=collection,
        config_path=PREFLIGHT_CONFIG_PATH,
    )
    print("Reused the completed immutable training preflight.")
else:
    preflight_report = run_capture_graph_training_preflight_v2(
        collection=collection,
        prepared_run_dir=PREPARED_RUN_DIR,
        graph_input_audit_dir=GRAPH_INPUT_AUDIT_DIR,
        config_path=PREFLIGHT_CONFIG_PATH,
        output_dir=PREFLIGHT_OUTPUT_DIR,
        local_work_root=LOCAL_PREFLIGHT_WORK_ROOT,
    )
    print("Saved the immutable training preflight.")

## 5. Review candidate blocks and selected boundaries

Each scenario independently selects the smallest declared block fraction that has an admissible block, then uses its latest admissible position. Verify both packet labels, benign-only and attack-containing windows, complete iterations, multiple steps, and the size of the suffix excluded during epoch selection.

In [ ]:
candidate_table = pd.read_csv(PREFLIGHT_OUTPUT_DIR / "split_candidates.csv")
try:
    split_table = pd.read_csv(PREFLIGHT_OUTPUT_DIR / "split_summary.csv")
except pd.errors.EmptyDataError:
    # Older blocked preflight runs wrote this artifact without a header.
    split_table = pd.DataFrame()
display(candidate_table)
print("Preflight status:", preflight_report["status"])
print("Selected validation block fraction by scenario:", json.dumps(
    preflight_report["selected_validation_block_fraction_by_scenario"],
    indent=2,
))
print("Fold totals:", json.dumps(preflight_report["fold_totals"], indent=2))
print("Checkpointing contract:", json.dumps(
    preflight_report["checkpointing"], indent=2
))
print("Final-refit contract:", json.dumps(
    preflight_report["final_refit"], indent=2
))
if split_table.empty:
    print("No complete set of scenario blocks was selected. Review the rejection reasons above.")
else:
    display(split_table)

## 6. Review the synthetic model-interface contract

Every variant must produce one finite logit per synthetic edge without mutating shared inputs. Temporal variants must create and then completely reset their state. No backward pass or optimizer is used.

In [ ]:
model_contract = preflight_report["model_contract"]
model_rows = [
    {"variant": name, **result}
    for name, result in model_contract["variants"].items()
]
display(pd.DataFrame(model_rows))
print("Shared inputs unchanged:", model_contract["shared_inputs_unchanged"])
print("Optimization performed:", model_contract["optimization_performed"])
print("Held-out scenarios accessed:",
      preflight_report["held_out_scenarios_accessed"])
print("Model training performed:",
      preflight_report["model_training_performed"])
print("Selection features materialized:",
      preflight_report["selection_features_materialized"])
print("Final refit performed:",
      preflight_report["final_refit_performed"])

## 7. Record the manual preflight decision

Approval freezes the scenario blocks, the per-model/per-fold best-epoch rule, and the complete-data final-refit rule. It authorizes creation of the full training runner but does not authorize training yet. Do not approve a blocked preflight.

In [ ]:
APPROVE_TRAINING_PREFLIGHT = False
PREFLIGHT_REVIEW_NOTE = "Unapproved"
DECISION_PATH = PREFLIGHT_OUTPUT_DIR / "training_preflight_decision.json"

if DECISION_PATH.exists():
    decision = json.loads(DECISION_PATH.read_text(encoding="utf-8"))
    print("Existing preflight decision:", json.dumps(decision, indent=2))
else:
    if not APPROVE_TRAINING_PREFLIGHT:
        raise RuntimeError(
            "Review the preflight, then explicitly approve or retain the run as unapproved."
        )
    decision = save_training_preflight_v2_decision(
        PREFLIGHT_OUTPUT_DIR,
        approved=True,
        review_note=PREFLIGHT_REVIEW_NOTE,
    )
    print("Saved preflight decision:", json.dumps(decision, indent=2))